In [5]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import joblib 
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, VotingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time

# Cell 2: Configurations
CLEAN_FILE_PATH = 'data_dongmai_clean.csv'
TARGET_COLUMN = 'pm2_5'
MODEL_EXPORT_PATH = 'ensemble_model_pm25.pkl'

# Định nghĩa các biến môi trường gốc cần tính Rolling
BASE_MET_FEATURES = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_u', 'wind_v']

# Cell 3: Core Evaluation Functions
def calculate_ioa(y_true, y_pred):
    mean_obs = np.mean(y_true)
    numerator = np.sum((y_pred - y_true)**2)
    denominator = np.sum((np.abs(y_pred - mean_obs) + np.abs(y_true - mean_obs))**2)
    return 1 - (numerator / denominator)

def evaluate_model(model_name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    ioa = calculate_ioa(y_true.values, y_pred)
    
    print(f"\n--- KẾT QUẢ: {model_name} ---")
    print(f"1. RMSE : {rmse:.4f}")
    print(f"2. MAE  : {mae:.4f}")
    print(f"3. R²   : {r2:.4f} (Mục tiêu >= 0.80)")
    print(f"4. IOA  : {ioa:.4f} (Phản ánh đường cong đồng điệu)")
    return r2

def export_saved_model(model, scaler, feature_names, filepath):
    """Lưu mô hình cùng các siêu tham số tiền xử lý"""
    try:
        model_data = {
            'model': model,
            'scaler': scaler,
            'feature_names': feature_names
        }
        joblib.dump(model_data, filepath)
        print(f"\n[THÀNH CÔNG] Siêu mô hình AI Ghi nhớ đã được lưu tại: {filepath}")
    except Exception as e:
        print(f"\n[LỖI] Không thể lưu: {e}")

# Cell 4: Core Analytics & Export Functions
def analyze_feature_importance(model, feature_names):
    rf_model = model.named_estimators_['rf']
    importances = rf_model.feature_importances_
    df_imp = pd.DataFrame({'Feature': feature_names, 'Importance (%)': importances * 100})
    df_imp = df_imp.sort_values(by='Importance (%)', ascending=False).reset_index(drop=True)
    
    print("\n--- PHÂN TÍCH ĐỘ QUAN TRỌNG CỦA BIẾN (FEATURE IMPORTANCE) ---")
    
    # CÁCH 1: Ép Pandas hiển thị toàn bộ dòng, không cắt bớt bằng dấu ...
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print(df_imp.to_string(index=False))

from IPython.display import display # Thêm thư viện này ở đầu file nếu chưa có

def analyze_residuals(y_true, y_pred):
    """
    Phân tích và thống kê độ lệch giữa dự báo và thực tế (Residual Analysis).
    Sử dụng DataFrame để hiển thị bảng chuẩn đẹp trên Google Colab.
    """
    y_true_np = y_true.values if isinstance(y_true, pd.Series) else y_true
    residuals = y_pred - y_true_np
    
    bins = [-np.inf, -20, -10, -5, 0, 5, 10, 20, np.inf]
    labels = [
        "Hụt nặng (<-20)", 
        "Hụt vừa (-20 đến -10)", 
        "Hụt nhẹ (-10 đến -5)", 
        "Hụt rất nhẹ (-5 đến 0)", 
        "Lố rất nhẹ (0 đến 5)", 
        "Lố nhẹ (5 đến 10)", 
        "Lố vừa (10 đến 20)", 
        "Lố nặng (>20)"
    ]
    
    cats = pd.cut(residuals, bins=bins, labels=labels)
    # Kế thừa cách sửa lỗi xuất sắc của bạn
    distribution = pd.Series(cats).value_counts(sort=False)
    
    total_samples = len(residuals)
    percentages = (distribution / total_samples) * 100
    
    # Tạo DataFrame để Colab vẽ bảng HTML tự động
    df_residuals = pd.DataFrame({
        'Khoảng Sai Số': labels,
        'Số lượng': distribution.values,
        'Tỷ lệ (%)': percentages.values.round(2) # Làm tròn 2 chữ số thập phân
    })
    
    print("\n" + "="*70)
    print(f"{'THỐNG KÊ PHÂN PHỐI SAI SỐ (RESIDUAL ANALYSIS)':^70}")
    print("="*70)
    
    # Ép Colab hiển thị bảng nguyên vẹn
    display(df_residuals)
    
    under_est = sum(residuals < 0)
    over_est = sum(residuals > 0)
    
    print("-" * 70)
    print(f"[*] Tổng số lần dự đoán HỤT (< 0) : {under_est} lần ({under_est/total_samples*100:.2f}%)")
    print(f"[*] Tổng số lần dự đoán LỐ (> 0)  : {over_est} lần ({over_est/total_samples*100:.2f}%)")
    print(f"[*] Cú đoán HỤT lớn nhất          : {residuals.min():.2f} µg/m³")
    print(f"[*] Cú đoán LỐ lớn nhất           : {residuals.max():.2f} µg/m³")
    print("="*70 + "\n")

# Cell 5: Main Execution Pipeline
if __name__ == "__main__":
    print("BẮT ĐẦU HUẤN LUYỆN: KẾT HỢP TẤT CẢ PHƯƠNG PHÁP & TRÍ NHỚ ĐẶC TRƯNG TÍCH LŨY (METHOD 4)")
    
    # =====================================================================
    # 1. TIỀN XỬ LÝ DỮ LIỆU: TRÍCH XUẤT ĐẶC TRƯNG TÍCH LŨY (ROLLING MEAN)
    # =====================================================================
    print("\nĐang thiết lập trí nhớ cho AI bằng cách trích xuất Gió/Nhiệt trung bình 12h-24h qua...")
    df = pd.read_csv(CLEAN_FILE_PATH)
    
    # Sắp xếp dữ liệu theo thời gian thực (đảm bảo rolling đúng)
    # Vì data clean đã có hour/month, ta cần tạo lại index time để rolling
    df_with_rolling = df.copy()
    
    # Thiết lập Rolling window 12h và 24h
    for feature in BASE_MET_FEATURES:
        df_with_rolling[f'{feature}_rolling_12h'] = df_with_rolling[feature].rolling(window=12, min_periods=1).mean()
        df_with_rolling[f'{feature}_rolling_24h'] = df_with_rolling[feature].rolling(window=24, min_periods=1).mean()
    
    # Lấy danh sách biến mới sau khi trích xuất
    X = df_with_rolling.drop(columns=[TARGET_COLUMN])
    y = df_with_rolling[TARGET_COLUMN]
    feature_names = X.columns.tolist()
    
    print(f"Số lượng đặc trưng đã tăng từ {len(df.columns)-1} lên {len(feature_names)} biến tích lũy.")
    # =====================================================================

    # 2. Phân chia dữ liệu
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
    
    # 3. KỸ THUẬT OVERSAMPLING & SAMPLE WEIGHTS (Kết hợp cả hai để tạo lực đẩy)
    sample_weights = np.ones(len(y_train))
    sample_weights[y_train > 35] = 1.5  
    sample_weights[y_train > 50] = 3.0
    # sample_weights[y_train > 80] = 5.0 # Mức 5.0 là đủ để chú ý, không làm AI bị "hoang tưởng"
    sample_weights[y_train > 80] = 10.0

   # =====================================================================
    # 4. KHỞI TẠO LIÊN MINH AI HỌC XẾP CHỒNG (STACKING REGRESSOR)
    # =====================================================================
    print("\nĐang khởi tạo liên minh AI TỐI ƯU HÓA ĐẶC TRƯNG (Mô hình Stacking)...")
    
    # Random Forest (Lớp 1): Cho phép cây phát triển tối đa để bắt mọi tín hiệu nhiễu nhỏ
    rf = RandomForestRegressor(
        n_estimators=100, 
        max_depth=None, 
        random_state=42, 
        n_jobs=-1
    )
    
    # XGBoost (Lớp 1): Áp dụng chiến thuật "Học nhiều bề rộng, giảm bề sâu" để chống Overfitting
    xgb = XGBRegressor(
        n_estimators=800,        
        learning_rate=0.05,      
        max_depth=8,             
        min_child_weight=3,      
        subsample=0.9,           
        colsample_bytree=0.8,    
        random_state=42,
        n_jobs=-1
    )
    
    # Meta-model (Lớp 2): Ridge Regression
    # Mô hình này sẽ phân tích sai số của RF và XGBoost để biết khi nào nên tin ai, 
    # đặc biệt là tự động bù trừ điểm yếu để không bị hụt các đỉnh ô nhiễm.
    meta_model = Ridge(alpha=1.0)
    
    # Sử dụng StackingRegressor thay cho VotingRegressor
    ensemble_model = StackingRegressor(
        estimators=[('rf', rf), ('xgb', xgb)],
        final_estimator=meta_model,
        cv=5, # Cắt dữ liệu thành 5 phần để Lớp 2 học cách sửa lỗi nội bộ
        n_jobs=-1
    )

    # 5. Huấn luyện (Sử dụng trí nhớ Rolling và trọng số Sample)
    print("Đang huấn luyện cỗ máy giới hạn (Stacking sẽ mất thời gian hơn Voting một chút)...")
    ensemble_model.fit(X_train, y_train, sample_weight=sample_weights)
    
    # 6. Đánh giá (Vượt mục tiêu)
    y_pred = ensemble_model.predict(X_test)
    evaluate_model("Ensemble (Stacking Regressor + Tuned Hyperparameters)", y_test, y_pred)

    # BỔ SUNG GỌI HÀM PHÂN TÍCH SAI SỐ Ở ĐÂY:
    analyze_residuals(y_test, y_pred)
    
    # 7. Phân tích & Đóng gói
    analyze_feature_importance(ensemble_model, feature_names)
    
    # Cần lưu cả Feature Names để app tiền xử lý
    export_saved_model(ensemble_model, None, feature_names, MODEL_EXPORT_PATH)

BẮT ĐẦU HUẤN LUYỆN: KẾT HỢP TẤT CẢ PHƯƠNG PHÁP & TRÍ NHỚ ĐẶC TRƯNG TÍCH LŨY (METHOD 4)

Đang thiết lập trí nhớ cho AI bằng cách trích xuất Gió/Nhiệt trung bình 12h-24h qua...
Số lượng đặc trưng đã tăng từ 13 lên 23 biến tích lũy.

Đang khởi tạo liên minh AI TỐI ƯU HÓA ĐẶC TRƯNG (Mô hình Stacking)...
Đang huấn luyện cỗ máy giới hạn (Stacking sẽ mất thời gian hơn Voting một chút)...

--- KẾT QUẢ: Ensemble (Stacking Regressor + Tuned Hyperparameters) ---
1. RMSE : 4.7741
2. MAE  : 3.0169
3. R²   : 0.9602 (Mục tiêu >= 0.80)
4. IOA  : 0.9897 (Phản ánh đường cong đồng điệu)

            THỐNG KÊ PHÂN PHỐI SAI SỐ (RESIDUAL ANALYSIS)             


,Khoảng Sai Số,Số lượng,Tỷ lệ (%)
0,Hụt nặng (<-20),10,0.18
1,Hụt vừa (-20 đến -10),75,1.32
2,Hụt nhẹ (-10 đến -5),229,4.02
3,Hụt rất nhẹ (-5 đến 0),1808,31.76
4,Lố rất nhẹ (0 đến 5),2882,50.62
5,Lố nhẹ (5 đến 10),542,9.52
6,Lố vừa (10 đến 20),134,2.35
7,Lố nặng (>20),13,0.23


----------------------------------------------------------------------
[*] Tổng số lần dự đoán HỤT (< 0) : 2122 lần (37.27%)
[*] Tổng số lần dự đoán LỐ (> 0)  : 3571 lần (62.73%)
[*] Cú đoán HỤT lớn nhất          : -97.85 µg/m³
[*] Cú đoán LỐ lớn nhất           : 58.70 µg/m³


--- PHÂN TÍCH ĐỘ QUAN TRỌNG CỦA BIẾN (FEATURE IMPORTANCE) ---
                         Feature  Importance (%)
                 carbon_monoxide       68.845558
                nitrogen_dioxide       14.862711
           aerosol_optical_depth        3.367804
                 sulphur_dioxide        1.521336
              wind_u_rolling_24h        1.329940
relative_humidity_2m_rolling_24h        1.270094
                           ozone        1.108659
              wind_u_rolling_12h        1.089887
              wind_v_rolling_24h        0.877084
                           month        0.857320
      temperature_2m_rolling_24h        0.747707
                  temperature_2m        0.551065
            relative_hu